# SH17 EfficientDet-D0 PyTorch Kaggle Benchmark

Notebook này là bản refactor sạch để chạy trên Kaggle cho bài toán PPE detection SH17.

Mục tiêu:

- Dùng PyTorch EfficientDet-D0 thay cho notebook TensorFlow cũ đã không còn hợp runtime Kaggle hiện tại.
- Đọc trực tiếp ảnh và YOLO labels từ `/kaggle/input`.
- Không tạo file trung gian lớn; checkpoint và metric ghi vào `/kaggle/working`.
- Có đủ baseline và 3 variant để so sánh với bảng YOLOv9s trong slide.
- Có checkpoint/resume, log từng epoch, early stopping, và bảng P/R/mAP50/mAP50-95.

Không bấm `Run All` ngay từ đầu. Hãy chạy lần lượt setup, data check, smoke test, rồi baseline pilot.


## 0. Baseline And Variant Map

| Section | Experiment | Ý nghĩa | Output |
| --- | --- | --- | --- |
| 7.1 | `effdet_d0_baseline_512` | Baseline nhẹ nhất, resize 512 + flip | `OUTPUT_ROOT/effdet_d0_baseline_512/` |
| 7.2 | `effdet_d0_tuned_512_es` | Variant 1: lr nhẹ hơn + early stopping + color jitter | `OUTPUT_ROOT/effdet_d0_tuned_512_es/` |
| 7.3 | `effdet_d0_oversample_minority_512_es` | Variant 2: oversample class PPE hiếm | `OUTPUT_ROOT/effdet_d0_oversample_minority_512_es/` |
| 7.4 | `effdet_d0_zoom_crop_512_es` | Variant 3: random resize/crop hỗ trợ object nhỏ | `OUTPUT_ROOT/effdet_d0_zoom_crop_512_es/` |

Thứ tự nên chạy:

1. Baseline debug hoặc pilot.
2. Variant 1 pilot.
3. Variant 2 nếu recall class hiếm thấp.
4. Variant 3 nếu AP object nhỏ thấp.
5. Final evaluation và so sánh bảng.


In [1]:
import kagglehub
from pathlib import Path

DATA_ROOT = Path(
    kagglehub.dataset_download(
        "mugheesahmad/sh17-dataset-for-ppe-detection"
    )
)

IMAGE_DIR = DATA_ROOT / "images"
LABEL_DIR = DATA_ROOT / "labels"
TRAIN_MANIFEST = DATA_ROOT / "train_files.txt"
VAL_MANIFEST = DATA_ROOT / "val_files.txt"

assert DATA_ROOT.exists(), f"Không tìm thấy dataset: {DATA_ROOT}"
assert IMAGE_DIR.exists(), f"Không tìm thấy images: {IMAGE_DIR}"
assert LABEL_DIR.exists(), f"Không tìm thấy labels: {LABEL_DIR}"
assert TRAIN_MANIFEST.exists(), f"Không tìm thấy train_files.txt"
assert VAL_MANIFEST.exists(), f"Không tìm thấy val_files.txt"

print("Dataset đã sẵn sàng:", DATA_ROOT)

100%|██████████| 13.1G/13.1G [05:33<00:00, 42.2MB/s]

Extracting files...


Dataset đã sẵn sàng: /root/.cache/kagglehub/datasets/mugheesahmad/sh17-dataset-for-ppe-detection/versions/1


## 1. Environment Setup

In [2]:
# Cài thư viện PyTorch EfficientDet.


!pip -q install "effdet==0.4.1" "timm>=0.9,<1.1" "pycocotools>=2.0.7" "pandas>=2.0.0" "pillow>=10.0.0" "tqdm>=4.66.0"

import json
import math
import os
import random
import shutil
import time
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageEnhance
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from tqdm.auto import tqdm

from effdet import create_model
from effdet.bench import DetBenchTrain, DetBenchPredict
from effdet.data.transforms import (
    Compose,
    ImageToNumpy,
    RandomFlip,
    RandomResizePad,
    ResizePad,
    IMAGENET_DEFAULT_MEAN,
    IMAGENET_DEFAULT_STD,
    resolve_fill_color,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.5/112.5 kB 7.8 MB/s eta 0:00:00
torch: 2.11.0+cu128
cuda: True
gpu: Tesla T4


In [3]:
# Cấu hình đường dẫn dành cho Google Colab.
# DATA_ROOT đã được tạo ở cell tải dataset bằng KaggleHub.

DATA_ROOT = Path(DATA_ROOT)

IMAGE_DIR = DATA_ROOT / "images"
LABEL_DIR = DATA_ROOT / "labels"
TRAIN_MANIFEST = DATA_ROOT / "train_files.txt"
VAL_MANIFEST = DATA_ROOT / "val_files.txt"

# Lưu checkpoint, metric và CSV trên ổ đĩa tạm của Colab.
OUTPUT_ROOT = Path("/content/SH17_outputs/effdet_d0_pytorch")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

assert DATA_ROOT.exists(), f"Không tìm thấy dataset: {DATA_ROOT}"
assert IMAGE_DIR.exists(), f"Không tìm thấy images: {IMAGE_DIR}"
assert LABEL_DIR.exists(), f"Không tìm thấy labels: {LABEL_DIR}"
assert TRAIN_MANIFEST.exists(), f"Không tìm thấy train_files.txt: {TRAIN_MANIFEST}"
assert VAL_MANIFEST.exists(), f"Không tìm thấy val_files.txt: {VAL_MANIFEST}"

def print_working_disk_usage(prefix="disk"):
    usage = shutil.disk_usage("/content")
    result = {
        "root": "/content",
        "used_gb": usage.used / (1024 ** 3),
        "free_gb": usage.free / (1024 ** 3),
        "total_gb": usage.total / (1024 ** 3),
    }

    print(
        f"{prefix}: used={result['used_gb']:.2f}GB "
        f"free={result['free_gb']:.2f}GB "
        f"total={result['total_gb']:.2f}GB "
        f"root={result['root']}"
    )
    return result

print("DATA_ROOT:", DATA_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("Train manifest:", TRAIN_MANIFEST)
print("Validation manifest:", VAL_MANIFEST)
print_working_disk_usage("initial")

DATA_ROOT: /root/.cache/kagglehub/datasets/mugheesahmad/sh17-dataset-for-ppe-detection/versions/1
OUTPUT_ROOT: /content/SH17_outputs/effdet_d0_pytorch
Train manifest: /root/.cache/kagglehub/datasets/mugheesahmad/sh17-dataset-for-ppe-detection/versions/1/train_files.txt
Validation manifest: /root/.cache/kagglehub/datasets/mugheesahmad/sh17-dataset-for-ppe-detection/versions/1/val_files.txt
initial: used=60.50GB free=52.12GB total=112.64GB root=/content


{'root': '/content',
 'used_gb': 60.4982795715332,
 'free_gb': 52.1226921081543,
 'total_gb': 112.6365966796875}

## 2. SH17 Data Checks

In [4]:
# Class order phải khớp file YOLO labels của SH17.

CLASS_NAMES = [
    "person",
    "ear",
    "ear-mufs",
    "face",
    "face-guard",
    "face-mask",
    "foot",
    "tool",
    "glasses",
    "gloves",
    "helmet",
    "hands",
    "head",
    "medical-suit",
    "shoes",
    "safety-suit",
    "safety-vest",
]

NUM_CLASSES = len(CLASS_NAMES)
MINORITY_CLASS_IDS_ZERO_BASED = {2, 4, 6, 10, 13, 16}
MINORITY_REPEAT_FACTOR = 3

def read_manifest(path):
    image_paths = []
    for raw in Path(path).read_text().splitlines():
        raw = raw.strip()
        if not raw:
            continue
        image_path = Path(raw)
        if not image_path.is_absolute():
            image_path = DATA_ROOT / raw
        if not image_path.exists():
            image_path = IMAGE_DIR / image_path.name
        image_paths.append(image_path)
    return image_paths

def label_path_for_image(image_path):
    return LABEL_DIR / f"{Path(image_path).stem}.txt"

def inspect_image(image_path):
    with Image.open(image_path) as image:
        return image.size

def parse_yolo_label_yxyx(label_path, image_width, image_height, one_based=True):
    boxes_yxyx = []
    labels = []
    label_path = Path(label_path)
    if not label_path.exists():
        return np.zeros((0, 4), dtype=np.float32), np.zeros((0,), dtype=np.int64)

    for raw in label_path.read_text().splitlines():
        raw = raw.strip()
        if not raw:
            continue
        class_id, xc, yc, bw, bh = raw.split()[:5]
        class_id = int(class_id)
        label = class_id + 1 if one_based else class_id
        xc, yc, bw, bh = map(float, (xc, yc, bw, bh))

        x1 = (xc - bw / 2.0) * image_width
        y1 = (yc - bh / 2.0) * image_height
        x2 = (xc + bw / 2.0) * image_width
        y2 = (yc + bh / 2.0) * image_height

        x1 = max(0.0, min(float(image_width), x1))
        y1 = max(0.0, min(float(image_height), y1))
        x2 = max(0.0, min(float(image_width), x2))
        y2 = max(0.0, min(float(image_height), y2))

        if x2 > x1 and y2 > y1:
            boxes_yxyx.append([y1, x1, y2, x2])
            labels.append(label)

    return np.asarray(boxes_yxyx, dtype=np.float32), np.asarray(labels, dtype=np.int64)

train_images = read_manifest(TRAIN_MANIFEST)
val_images = read_manifest(VAL_MANIFEST)

def summarize_split(image_paths, split_name):
    class_counter = Counter()
    image_counter = Counter()
    missing_labels = 0
    for image_path in tqdm(image_paths, desc=f"checking {split_name}"):
        width, height = inspect_image(image_path)
        label_path = label_path_for_image(image_path)
        if not label_path.exists():
            missing_labels += 1
        _, labels_zero = parse_yolo_label_yxyx(label_path, width, height, one_based=False)
        class_counter.update(labels_zero.tolist())
        image_counter.update(set(labels_zero.tolist()))
    return {
        "split": split_name,
        "images": len(image_paths),
        "instances": sum(class_counter.values()),
        "missing_labels": missing_labels,
        "class_counter": class_counter,
        "image_counter": image_counter,
    }

train_summary = summarize_split(train_images, "train")
val_summary = summarize_split(val_images, "val")

def contains_minority(image_path):
    width, height = inspect_image(image_path)
    _, labels_zero = parse_yolo_label_yxyx(label_path_for_image(image_path), width, height, one_based=False)
    return bool(set(labels_zero.tolist()) & MINORITY_CLASS_IDS_ZERO_BASED)

minority_train_images = [path for path in tqdm(train_images, desc="minority check") if contains_minority(path)]

print("train images", train_summary["images"])
print("val images", val_summary["images"])
print("train instances", train_summary["instances"])
print("val instances", val_summary["instances"])
print("total instances", train_summary["instances"] + val_summary["instances"])
print("missing labels", train_summary["missing_labels"] + val_summary["missing_labels"])
print("minority train images", len(minority_train_images))


checking train:   0%|          | 0/6479 [00:00<?, ?it/s]

checking val:   0%|          | 0/1620 [00:00<?, ?it/s]

minority check:   0%|          | 0/6479 [00:00<?, ?it/s]

train images 6479
val images 1620
train instances 60636
val instances 15358
total instances 75994
missing labels 0
minority train images 1002


## 3. PyTorch Dataset And Augmentations

In [5]:
# Dataset và augmentation.
# effdet AnchorLabeler sử dụng bbox dạng [ymin, xmin, ymax, xmax]
# và class ID bắt đầu từ 1.

class RandomBrightnessContrast:
    def __init__(self, brightness=0.12, contrast=0.12, prob=0.5):
        self.brightness = brightness
        self.contrast = contrast
        self.prob = prob

    def __call__(self, img, annotations):
        if random.random() >= self.prob:
            return img, annotations

        brightness_factor = 1.0 + random.uniform(
            -self.brightness,
            self.brightness,
        )
        contrast_factor = 1.0 + random.uniform(
            -self.contrast,
            self.contrast,
        )

        img = ImageEnhance.Brightness(img).enhance(brightness_factor)
        img = ImageEnhance.Contrast(img).enhance(contrast_factor)

        return img, annotations


def make_transform(policy, image_size, training):
    fill_color = resolve_fill_color("mean", IMAGENET_DEFAULT_MEAN)
    transforms = []

    if training:
        transforms.append(
            RandomFlip(horizontal=True, prob=0.5)
        )

        if policy in {"tuned", "zoom_crop"}:
            transforms.append(
                RandomBrightnessContrast()
            )

        if policy == "zoom_crop":
            transforms.append(
                RandomResizePad(
                    target_size=image_size,
                    scale=(0.65, 1.45),
                    interpolation="random",
                    fill_color=fill_color,
                )
            )
        else:
            transforms.append(
                ResizePad(
                    target_size=image_size,
                    interpolation="bilinear",
                    fill_color=fill_color,
                )
            )
    else:
        transforms.append(
            ResizePad(
                target_size=image_size,
                interpolation="bilinear",
                fill_color=fill_color,
            )
        )

    transforms.append(ImageToNumpy())

    return Compose(transforms)


class SH17DetectionDataset(torch.utils.data.Dataset):
    def __init__(
        self,
        image_paths,
        image_size=512,
        augmentation_policy="baseline",
        training=False,
    ):
        self.image_paths = list(image_paths)
        self.image_size = int(image_size)
        self.training = bool(training)

        self.transform = make_transform(
            augmentation_policy,
            self.image_size,
            self.training,
        )

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, index):
        image_path = Path(self.image_paths[index])

        with Image.open(image_path) as opened_image:
            image = opened_image.convert("RGB")

        width, height = image.size

        boxes_yxyx, labels_one = parse_yolo_label_yxyx(
            label_path_for_image(image_path),
            width,
            height,
            one_based=True,
        )

        annotations = {
            "bbox": boxes_yxyx.astype(np.float32),
            "cls": labels_one.astype(np.float32),

            # ResizePad sẽ tự cập nhật img_scale.
            "img_scale": np.float32(1.0),

            # Kích thước ảnh gốc theo thứ tự [width, height].
            # Dùng để đưa prediction về kích thước ảnh gốc khi evaluation.
            "img_size": np.asarray(
                [width, height],
                dtype=np.float32,
            ),

            "image_id": np.int64(index),
            "source_index": np.int64(index),
        }

        image_np, annotations = self.transform(
            image,
            annotations,
        )

        annotations["image_path"] = str(image_path)

        return image_np, annotations


MEAN_255 = torch.tensor(
    [value * 255.0 for value in IMAGENET_DEFAULT_MEAN]
).view(1, 3, 1, 1)

STD_255 = torch.tensor(
    [value * 255.0 for value in IMAGENET_DEFAULT_STD]
).view(1, 3, 1, 1)


def collate_detection_batch(batch):
    images = torch.stack(
        [
            torch.from_numpy(item[0]).float()
            for item in batch
        ]
    )

    images = (images - MEAN_255) / STD_255

    target = {
        "bbox": [
            torch.as_tensor(
                item[1]["bbox"],
                dtype=torch.float32,
            )
            for item in batch
        ],
        "cls": [
            torch.as_tensor(
                item[1]["cls"],
                dtype=torch.float32,
            )
            for item in batch
        ],
        "img_scale": torch.as_tensor(
            [item[1]["img_scale"] for item in batch],
            dtype=torch.float32,
        ),
        "img_size": torch.stack(
            [
                torch.as_tensor(
                    item[1]["img_size"],
                    dtype=torch.float32,
                )
                for item in batch
            ]
        ),
        "image_id": torch.as_tensor(
            [item[1]["image_id"] for item in batch],
            dtype=torch.int64,
        ),
        "image_path": [
            item[1]["image_path"]
            for item in batch
        ],
    }

    return images, target


def move_target_to_device(target, device):
    moved = {}

    for key, value in target.items():
        if isinstance(value, list):
            moved[key] = [
                item.to(device) if torch.is_tensor(item) else item
                for item in value
            ]
        elif torch.is_tensor(value):
            moved[key] = value.to(device)
        else:
            moved[key] = value

    return moved


# Smoke test dataset.
sample_dataset = SH17DetectionDataset(
    train_images[:8],
    image_size=512,
    augmentation_policy="baseline",
    training=True,
)

sample_image, sample_target = sample_dataset[0]

print("sample image tensor:", torch.from_numpy(sample_image).shape)
print("sample boxes:", sample_target["bbox"].shape)
print("sample labels:", sample_target["cls"][:5])
print("original image size:", sample_target["img_size"])
print("image scale:", sample_target["img_scale"])

assert sample_image.shape == (3, 512, 512)
assert sample_target["bbox"].ndim == 2
assert sample_target["bbox"].shape[1] == 4
assert len(sample_target["bbox"]) == len(sample_target["cls"])

print("Dataset smoke test thành công.")

sample image tensor: torch.Size([3, 512, 512])
sample boxes: (8, 4)
sample labels: [13.  4. 12. 12.  7.]
original image size: [6068. 4045.]
image scale: 11.851562500000002
Dataset smoke test thành công.


## 4. EfficientDet-D0 Model Factory

In [6]:
# Experiment definitions.

@dataclass(frozen=True)
class EffdetExperiment:
    name: str
    image_size: int = 512
    max_epochs: int = 50
    pilot_epochs: int = 20
    batch_size: int = 8
    lr: float = 2e-4
    weight_decay: float = 1e-4
    eval_every_epochs: int = 5
    early_stopping: bool = False
    early_stopping_patience_epochs: int = 10
    early_stopping_min_delta: float = 0.05
    use_oversampling: bool = False
    augmentation_policy: str = "baseline"

EXPERIMENTS = [
    EffdetExperiment("effdet_d0_baseline_512", pilot_epochs=20, batch_size=8, augmentation_policy="baseline"),
    EffdetExperiment("effdet_d0_tuned_512_es", pilot_epochs=30, batch_size=8, lr=1.5e-4, early_stopping=True, augmentation_policy="tuned"),
    EffdetExperiment("effdet_d0_oversample_minority_512_es", pilot_epochs=30, batch_size=8, lr=1.5e-4, early_stopping=True, use_oversampling=True, augmentation_policy="tuned"),
    EffdetExperiment("effdet_d0_zoom_crop_512_es", pilot_epochs=30, batch_size=8, lr=1.5e-4, early_stopping=True, augmentation_policy="zoom_crop"),
]

EXPERIMENT_LOOKUP = {experiment.name: experiment for experiment in EXPERIMENTS}
ACTIVE_EXPERIMENT_NAME = "effdet_d0_baseline_512"
RUN_EXPERIMENT_NAMES = [ACTIVE_EXPERIMENT_NAME]

def experiment_output_dir(experiment):
    path = OUTPUT_ROOT / experiment.name
    path.mkdir(parents=True, exist_ok=True)
    (path / "checkpoints").mkdir(parents=True, exist_ok=True)
    return path

def train_paths_for_experiment(experiment):
    if not experiment.use_oversampling:
        return train_images
    minority_set = set(minority_train_images)
    expanded = []
    for image_path in train_images:
        repeat = MINORITY_REPEAT_FACTOR if image_path in minority_set else 1
        expanded.extend([image_path] * repeat)
    return expanded

def make_loaders(experiment, debug_max_batches=None):
    train_paths = train_paths_for_experiment(experiment)
    train_dataset = SH17DetectionDataset(
        train_paths,
        image_size=experiment.image_size,
        augmentation_policy=experiment.augmentation_policy,
        training=True,
    )
    val_dataset = SH17DetectionDataset(
        val_images,
        image_size=experiment.image_size,
        augmentation_policy="baseline",
        training=False,
    )
    train_loader = torch.utils.data.DataLoader(
        train_dataset,
        batch_size=experiment.batch_size,
        shuffle=True,
        num_workers=2,
        pin_memory=DEVICE.type == "cuda",
        collate_fn=collate_detection_batch,
        drop_last=False,
    )
    val_loader = torch.utils.data.DataLoader(
        val_dataset,
        batch_size=max(1, experiment.batch_size),
        shuffle=False,
        num_workers=2,
        pin_memory=DEVICE.type == "cuda",
        collate_fn=collate_detection_batch,
        drop_last=False,
    )
    return train_loader, val_loader

def create_train_model(num_classes=NUM_CLASSES, image_size=512):
    model = create_model(
        "tf_efficientdet_d0",
        bench_task="train",
        bench_labeler=True,
        num_classes=num_classes,
        pretrained=True,
        image_size=(image_size, image_size),
    )
    return model

def create_predict_model(checkpoint_path, num_classes=NUM_CLASSES, image_size=512):
    model = create_model(
        "tf_efficientdet_d0",
        bench_task="predict",
        num_classes=num_classes,
        pretrained=False,
        pretrained_backbone=False,
        image_size=(image_size, image_size),
    )

    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False,
    )

    missing, unexpected = model.load_state_dict(
        checkpoint["model"],
        strict=False,
    )

    print(
        "load predict model missing:",
        len(missing),
        "unexpected:",
        len(unexpected),
    )

    model.eval()
    return model

def count_trainable_params(model):
    return sum(param.numel() for param in model.parameters() if param.requires_grad)

# Smoke test chỉ khởi tạo model. Chạy cell này trước baseline để kiểm tra import/model ok.
smoke_model = create_train_model(NUM_CLASSES, 512)
print("created EfficientDet-D0 train model")
print("trainable params:", count_trainable_params(smoke_model))
del smoke_model
torch.cuda.empty_cache()


Downloading: "https://github.com/rwightman/efficientdet-pytorch/releases/download/v0.1/tf_efficientdet_d0_34-f153e0cf.pth" to /root/.cache/torch/hub/checkpoints/tf_efficientdet_d0_34-f153e0cf.pth
created EfficientDet-D0 train model
trainable params: 3837362


## 5. Checkpoint And History Utilities

In [7]:
# Checkpoint, resume và history.

def checkpoint_paths(experiment):
    root = experiment_output_dir(experiment) / "checkpoints"
    return root / "last.pt", root / "best.pt"

def history_path(experiment):
    return experiment_output_dir(experiment) / "training_history.csv"

def load_history(experiment):
    path = history_path(experiment)
    if path.exists():
        return pd.read_csv(path)
    return pd.DataFrame()

def save_checkpoint(experiment, epoch, model, optimizer, scaler, best_map50_95, target_name="last.pt"):
    output_dir = experiment_output_dir(experiment)
    checkpoint_path = output_dir / "checkpoints" / target_name
    payload = {
        "experiment": experiment.name,
        "epoch": int(epoch),
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scaler": scaler.state_dict() if scaler is not None else None,
        "best_map50_95": float(best_map50_95),
        "class_names": CLASS_NAMES,
        "experiment_config": asdict(experiment),
    }
    tmp_path = checkpoint_path.with_suffix(".tmp")
    torch.save(payload, tmp_path)
    tmp_path.replace(checkpoint_path)
    print("saved checkpoint:", checkpoint_path)
    return checkpoint_path

def load_checkpoint_if_available(
    experiment,
    model,
    optimizer=None,
    scaler=None,
):
    last_path, _ = checkpoint_paths(experiment)

    if not last_path.exists():
        print("Không có checkpoint cũ, bắt đầu train từ epoch 1.")
        return 0, -1.0

    checkpoint = torch.load(
        last_path,
        map_location="cpu",
        weights_only=False,
    )

    model.load_state_dict(
        checkpoint["model"],
        strict=False,
    )
    model.to(DEVICE)

    if optimizer is not None and checkpoint.get("optimizer") is not None:
        optimizer.load_state_dict(checkpoint["optimizer"])

    if scaler is not None and checkpoint.get("scaler") is not None:
        scaler.load_state_dict(checkpoint["scaler"])

    print("resumed checkpoint:", last_path)

    return (
        int(checkpoint.get("epoch", 0)),
        float(checkpoint.get("best_map50_95", -1.0)),
    )

def append_history(experiment, row):
    path = history_path(experiment)
    history = load_history(experiment)
    history = pd.concat([history, pd.DataFrame([row])], ignore_index=True)
    history.to_csv(path, index=False)
    return history


## 6. Evaluation Helpers Used During Training

Các hàm này được đặt trước train loop để notebook chạy được từ trên xuống. Chúng tạo COCO ground truth, chạy predict, tính mAP50/mAP50-95 và quét threshold P/R/F1.

In [8]:
# COCO-style evaluation và precision/recall threshold sweep.

def yxyx_to_xywh(box):
    y1, x1, y2, x2 = map(float, box)
    return [
        x1,
        y1,
        max(0.0, x2 - x1),
        max(0.0, y2 - y1),
    ]


def yxyx_to_xyxy(box):
    y1, x1, y2, x2 = map(float, box)
    return [x1, y1, x2, y2]


def build_coco_ground_truth(image_paths):
    images = []
    annotations = []
    annotation_id = 1

    for image_id, image_path in enumerate(
        tqdm(image_paths, desc="Building COCO ground truth")
    ):
        width, height = inspect_image(image_path)

        boxes, labels = parse_yolo_label_yxyx(
            label_path_for_image(image_path),
            width,
            height,
            one_based=True,
        )

        images.append(
            {
                "id": image_id,
                "width": width,
                "height": height,
                "file_name": Path(image_path).name,
            }
        )

        for box, label in zip(boxes, labels):
            bbox = yxyx_to_xywh(box)

            annotations.append(
                {
                    "id": annotation_id,
                    "image_id": image_id,
                    "category_id": int(label),
                    "bbox": bbox,
                    "area": bbox[2] * bbox[3],
                    "iscrowd": 0,
                }
            )

            annotation_id += 1

    coco = COCO()

    coco.dataset = {
        "images": images,
        "annotations": annotations,
        "categories": [
            {
                "id": index + 1,
                "name": class_name,
                "supercategory": "ppe",
            }
            for index, class_name in enumerate(CLASS_NAMES)
        ],
    }

    coco.createIndex()
    return coco


# Ground truth chỉ cần tạo một lần.
coco_gt = build_coco_ground_truth(val_images)


def run_predictions(
    model,
    val_loader,
    max_batches=None,
    score_threshold=0.001,
    return_image_ids=False,
):
    # Khi evaluate model đang train, dùng phần model bên trong
    # để tạo prediction bench.
    if isinstance(model, DetBenchTrain):
        prediction_model = DetBenchPredict(model.model).to(DEVICE)
    else:
        prediction_model = model

    prediction_model.eval()

    predictions = []
    evaluated_image_ids = []

    with torch.inference_mode():
        for batch_index, (images, target) in enumerate(
            tqdm(val_loader, desc="Predicting"),
            start=1,
        ):
            images = images.to(DEVICE, non_blocking=True)

            image_info = {
                "img_scale": target["img_scale"].to(
                    DEVICE,
                    non_blocking=True,
                ),
                "img_size": target["img_size"].to(
                    DEVICE,
                    non_blocking=True,
                ),
            }

            detections = prediction_model(images, image_info)
            detections = detections.detach().cpu().numpy()

            image_ids = target["image_id"].cpu().numpy().tolist()
            evaluated_image_ids.extend(int(value) for value in image_ids)

            for row_index, image_id in enumerate(image_ids):
                for detection in detections[row_index]:
                    x1, y1, x2, y2, score, class_id = detection.tolist()

                    if score < score_threshold:
                        continue

                    if class_id < 1 or class_id > NUM_CLASSES:
                        continue

                    predictions.append(
                        {
                            "image_id": int(image_id),
                            "category_id": int(class_id),
                            "bbox": [
                                float(x1),
                                float(y1),
                                float(max(0.0, x2 - x1)),
                                float(max(0.0, y2 - y1)),
                            ],
                            "score": float(score),
                            "box_xyxy": [
                                float(x1),
                                float(y1),
                                float(x2),
                                float(y2),
                            ],
                        }
                    )

            if max_batches is not None and batch_index >= max_batches:
                break

    if return_image_ids:
        return predictions, evaluated_image_ids

    return predictions


def run_coco_eval(predictions, image_ids=None):
    if not predictions:
        return None, {
            "mAP50": 0.0,
            "mAP50-95": 0.0,
        }

    coco_predictions = [
        {
            "image_id": prediction["image_id"],
            "category_id": prediction["category_id"],
            "bbox": prediction["bbox"],
            "score": prediction["score"],
        }
        for prediction in predictions
    ]

    coco_detections = coco_gt.loadRes(coco_predictions)
    evaluator = COCOeval(coco_gt, coco_detections, "bbox")

    evaluator.params.catIds = list(range(1, NUM_CLASSES + 1))

    # Khi smoke test, chỉ đánh giá những ảnh đã được predict.
    if image_ids is not None:
        evaluator.params.imgIds = sorted(set(map(int, image_ids)))

    evaluator.evaluate()
    evaluator.accumulate()
    evaluator.summarize()

    return evaluator, {
        "mAP50": float(evaluator.stats[1]) * 100.0,
        "mAP50-95": float(evaluator.stats[0]) * 100.0,
    }


def box_iou_matrix(boxes_a, boxes_b):
    if len(boxes_a) == 0 or len(boxes_b) == 0:
        return np.zeros(
            (len(boxes_a), len(boxes_b)),
            dtype=np.float32,
        )

    boxes_a = np.asarray(boxes_a, dtype=np.float32)
    boxes_b = np.asarray(boxes_b, dtype=np.float32)

    area_a = (
        np.maximum(0, boxes_a[:, 2] - boxes_a[:, 0])
        * np.maximum(0, boxes_a[:, 3] - boxes_a[:, 1])
    )

    area_b = (
        np.maximum(0, boxes_b[:, 2] - boxes_b[:, 0])
        * np.maximum(0, boxes_b[:, 3] - boxes_b[:, 1])
    )

    left_top = np.maximum(
        boxes_a[:, None, :2],
        boxes_b[None, :, :2],
    )

    right_bottom = np.minimum(
        boxes_a[:, None, 2:],
        boxes_b[None, :, 2:],
    )

    width_height = np.maximum(0, right_bottom - left_top)
    intersection = width_height[:, :, 0] * width_height[:, :, 1]

    union = area_a[:, None] + area_b[None, :] - intersection

    return intersection / np.maximum(union, 1e-9)


def load_targets_for_pr(image_paths):
    targets = []

    for image_id, image_path in enumerate(
        tqdm(image_paths, desc="Loading PR targets")
    ):
        width, height = inspect_image(image_path)

        boxes_yxyx, labels = parse_yolo_label_yxyx(
            label_path_for_image(image_path),
            width,
            height,
            one_based=True,
        )

        boxes_xyxy = np.asarray(
            [yxyx_to_xyxy(box) for box in boxes_yxyx],
            dtype=np.float32,
        )

        targets.append(
            {
                "image_id": image_id,
                "boxes": boxes_xyxy,
                "labels": labels,
            }
        )

    return targets


targets_for_pr = load_targets_for_pr(val_images)


def match_predictions_at_threshold(
    predictions,
    targets,
    score_threshold,
    iou_threshold=0.5,
):
    targets_by_image = {
        item["image_id"]: item
        for item in targets
    }

    predictions_by_image = defaultdict(list)

    for prediction in predictions:
        if prediction["score"] >= score_threshold:
            predictions_by_image[prediction["image_id"]].append(
                prediction
            )

    totals = {
        class_id: {
            "tp": 0,
            "fp": 0,
            "fn": 0,
        }
        for class_id in range(1, NUM_CLASSES + 1)
    }

    for image_id, target in targets_by_image.items():
        target_boxes = np.asarray(
            target["boxes"],
            dtype=np.float32,
        )

        target_labels = np.asarray(
            target["labels"],
            dtype=np.int64,
        )

        image_predictions = sorted(
            predictions_by_image.get(image_id, []),
            key=lambda item: item["score"],
            reverse=True,
        )

        for class_id in totals:
            ground_truth_boxes = target_boxes[
                target_labels == class_id
            ]

            predicted_boxes = np.asarray(
                [
                    prediction["box_xyxy"]
                    for prediction in image_predictions
                    if prediction["category_id"] == class_id
                ],
                dtype=np.float32,
            )

            matched_ground_truth = set()

            ious = box_iou_matrix(
                predicted_boxes,
                ground_truth_boxes,
            )

            for prediction_index in range(len(predicted_boxes)):
                if len(ground_truth_boxes) == 0:
                    totals[class_id]["fp"] += 1
                    continue

                best_gt_index = int(
                    np.argmax(ious[prediction_index])
                )

                best_iou = float(
                    ious[prediction_index, best_gt_index]
                )

                if (
                    best_iou >= iou_threshold
                    and best_gt_index not in matched_ground_truth
                ):
                    totals[class_id]["tp"] += 1
                    matched_ground_truth.add(best_gt_index)
                else:
                    totals[class_id]["fp"] += 1

            totals[class_id]["fn"] += max(
                0,
                len(ground_truth_boxes) - len(matched_ground_truth),
            )

    total_tp = sum(item["tp"] for item in totals.values())
    total_fp = sum(item["fp"] for item in totals.values())
    total_fn = sum(item["fn"] for item in totals.values())

    precision = total_tp / max(1, total_tp + total_fp)
    recall = total_tp / max(1, total_tp + total_fn)
    f1 = 2 * precision * recall / max(1e-9, precision + recall)

    return (
        precision * 100.0,
        recall * 100.0,
        f1 * 100.0,
    )


def best_pr_threshold(predictions, targets):
    thresholds = np.concatenate(
        [
            np.linspace(0.001, 0.05, 10),
            np.linspace(0.06, 0.50, 23),
            np.linspace(0.55, 0.95, 9),
        ]
    )

    rows = []

    for threshold in thresholds:
        precision, recall, f1 = match_predictions_at_threshold(
            predictions,
            targets,
            float(threshold),
        )

        rows.append(
            {
                "threshold": float(threshold),
                "P": precision,
                "R": recall,
                "F1": f1,
            }
        )

    dataframe = pd.DataFrame(rows)

    return (
        dataframe.sort_values(
            "F1",
            ascending=False,
        ).iloc[0].to_dict(),
        dataframe,
    )


def per_class_metrics_from_coco(evaluator):
    if evaluator is None:
        return pd.DataFrame(
            {
                "class": CLASS_NAMES,
                "AP50": [0.0] * NUM_CLASSES,
                "AP50-95": [0.0] * NUM_CLASSES,
            }
        )

    precision = evaluator.eval["precision"]
    iou_thresholds = evaluator.params.iouThrs
    rows = []

    iou50_index = int(
        np.where(np.isclose(iou_thresholds, 0.5))[0][0]
    )

    for class_index, class_name in enumerate(CLASS_NAMES):
        class_precision = precision[:, :, class_index, 0, -1]
        valid_precision = class_precision[class_precision > -1]

        ap_all = (
            float(np.mean(valid_precision)) * 100.0
            if valid_precision.size
            else 0.0
        )

        class_precision50 = precision[
            iou50_index,
            :,
            class_index,
            0,
            -1,
        ]

        valid_precision50 = class_precision50[
            class_precision50 > -1
        ]

        ap50 = (
            float(np.mean(valid_precision50)) * 100.0
            if valid_precision50.size
            else 0.0
        )

        rows.append(
            {
                "class": class_name,
                "AP50": ap50,
                "AP50-95": ap_all,
            }
        )

    return pd.DataFrame(rows)


def evaluate_model_checkpointless(
    model,
    val_loader,
    experiment,
    max_batches=None,
):
    predictions, evaluated_image_ids = run_predictions(
        model,
        val_loader,
        max_batches=max_batches,
        return_image_ids=True,
    )

    evaluator, coco_metrics = run_coco_eval(
        predictions,
        image_ids=evaluated_image_ids,
    )

    evaluated_id_set = set(evaluated_image_ids)

    evaluated_targets = [
        target
        for target in targets_for_pr
        if target["image_id"] in evaluated_id_set
    ]

    best_pr, _ = best_pr_threshold(
        predictions,
        evaluated_targets,
    )

    return {
        "mAP50": coco_metrics["mAP50"],
        "mAP50-95": coco_metrics["mAP50-95"],
        "P": best_pr["P"],
        "R": best_pr["R"],
        "F1": best_pr["F1"],
        "best_conf_threshold": best_pr["threshold"],
    }


print("Evaluation pipeline đã sẵn sàng.")

Building COCO ground truth:   0%|          | 0/1620 [00:00<?, ?it/s]

creating index...
index created!


Loading PR targets:   0%|          | 0/1620 [00:00<?, ?it/s]

Evaluation pipeline đã sẵn sàng.


In [9]:
# Final evaluation một experiment từ best.pt hoặc last.pt.

def evaluate_experiment_final(
    experiment,
    checkpoint_name="best.pt",
):
    output_dir = experiment_output_dir(experiment)

    checkpoint_path = (
        output_dir
        / "checkpoints"
        / checkpoint_name
    )

    # Nếu chưa có best.pt thì sử dụng last.pt.
    if not checkpoint_path.exists():
        checkpoint_path = (
            output_dir
            / "checkpoints"
            / "last.pt"
        )

    if not checkpoint_path.exists():
        raise FileNotFoundError(
            f"Không thấy checkpoint để evaluate: {checkpoint_path}"
        )

    print("Evaluating checkpoint:", checkpoint_path)

    _, val_loader = make_loaders(experiment)

    model = create_predict_model(
        checkpoint_path,
        NUM_CLASSES,
        experiment.image_size,
    ).to(DEVICE)

    predictions, evaluated_image_ids = run_predictions(
        model,
        val_loader,
        score_threshold=0.001,
        return_image_ids=True,
    )

    # Lưu toàn bộ prediction để có thể kiểm tra lại sau này.
    predictions_path = output_dir / "eval_predictions.json"

    with predictions_path.open("w", encoding="utf-8") as file:
        json.dump(predictions, file)

    evaluator, coco_metrics = run_coco_eval(
        predictions,
        image_ids=evaluated_image_ids,
    )

    evaluated_id_set = set(evaluated_image_ids)

    evaluated_targets = [
        target
        for target in targets_for_pr
        if target["image_id"] in evaluated_id_set
    ]

    pr_best, pr_curve = best_pr_threshold(
        predictions,
        evaluated_targets,
    )

    per_class = per_class_metrics_from_coco(evaluator)

    pr_curve.to_csv(
        output_dir / "pr_curve.csv",
        index=False,
    )

    per_class.to_csv(
        output_dir / "per_class_metrics.csv",
        index=False,
    )

    overall = {
        "model": "EfficientDet-D0 PyTorch",
        "variant": experiment.name,
        "checkpoint": checkpoint_path.name,
        "input_size": experiment.image_size,
        "train_entries": len(
            train_paths_for_experiment(experiment)
        ),
        "validation_images": len(evaluated_image_ids),
        "P": pr_best["P"],
        "R": pr_best["R"],
        "F1": pr_best["F1"],
        "best_conf_threshold": pr_best["threshold"],
        "mAP50": coco_metrics["mAP50"],
        "mAP50-95": coco_metrics["mAP50-95"],
    }

    overall_dataframe = pd.DataFrame([overall])

    overall_dataframe.to_csv(
        output_dir / "overall_metrics.csv",
        index=False,
    )

    display(overall_dataframe.round(3))
    display(per_class.round(3))

    del model
    torch.cuda.empty_cache()

    return overall, per_class


def build_overall_comparison(experiment_names):
    rows = []

    for experiment_name in experiment_names:
        experiment = EXPERIMENT_LOOKUP[experiment_name]

        metrics_path = (
            experiment_output_dir(experiment)
            / "overall_metrics.csv"
        )

        if metrics_path.exists():
            rows.append(
                pd.read_csv(metrics_path).iloc[0].to_dict()
            )
        else:
            print(
                f"Chưa có overall_metrics.csv cho: {experiment_name}"
            )

    comparison = pd.DataFrame(rows)

    if comparison.empty:
        print("Chưa có experiment nào được final evaluation.")
        return comparison

    comparison.to_csv(
        OUTPUT_ROOT / "overall_comparison.csv",
        index=False,
    )

    display(comparison.round(3))

    return comparison


print("Final evaluation functions đã sẵn sàng.")

# Chỉ chạy sau khi đã train xong:
# overall, per_class = evaluate_experiment_final(
#     EXPERIMENT_LOOKUP["effdet_d0_baseline_512"]
# )

# comparison = build_overall_comparison([
#     "effdet_d0_baseline_512",
#     "effdet_d0_tuned_512_es",
#     "effdet_d0_oversample_minority_512_es",
#     "effdet_d0_zoom_crop_512_es",
# ])

Final evaluation functions đã sẵn sàng.


## 7. Train EfficientDet-D0 Variants


In [10]:
# Train loop theo từng epoch.

# None: train đầy đủ.
# Ví dụ 20: smoke test 20 batch, chỉ chạy 1 epoch và KHÔNG lưu checkpoint.
DEBUG_MAX_BATCHES = None


def train_one_epoch(
    model,
    loader,
    optimizer,
    scaler,
    epoch,
    experiment,
    debug_max_batches=None,
):
    model.train()

    total_loss = 0.0
    total_class_loss = 0.0
    total_box_loss = 0.0
    num_batches = 0

    progress = tqdm(
        loader,
        desc=f"{experiment.name} - epoch {epoch}",
    )

    for batch_index, (images, target) in enumerate(
        progress,
        start=1,
    ):
        images = images.to(
            DEVICE,
            non_blocking=True,
        )

        target = move_target_to_device(
            target,
            DEVICE,
        )

        optimizer.zero_grad(set_to_none=True)

        # AMP giúp train nhanh hơn và giảm VRAM trên Tesla T4.
        with torch.amp.autocast(
            device_type=DEVICE.type,
            enabled=USE_AMP,
        ):
            output = model(images, target)
            loss = output["loss"]

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Loss không hợp lệ tại epoch {epoch}, "
                f"batch {batch_index}: {loss.item()}"
            )

        if USE_AMP:
            scaler.scale(loss).backward()

            # Unscale trước khi gradient clipping.
            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=10.0,
            )

            scaler.step(optimizer)
            scaler.update()

        else:
            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=10.0,
            )

            optimizer.step()

        loss_value = float(loss.detach().cpu())

        class_loss = float(
            output["class_loss"].detach().cpu()
        )

        box_loss = float(
            output["box_loss"].detach().cpu()
        )

        total_loss += loss_value
        total_class_loss += class_loss
        total_box_loss += box_loss
        num_batches += 1

        progress.set_postfix(
            loss=f"{loss_value:.4f}",
            cls=f"{class_loss:.4f}",
            box=f"{box_loss:.4f}",
        )

        if (
            debug_max_batches is not None
            and batch_index >= debug_max_batches
        ):
            break

    denominator = max(1, num_batches)

    return {
        "train_loss": total_loss / denominator,
        "class_loss": total_class_loss / denominator,
        "box_loss": total_box_loss / denominator,
        "train_batches": num_batches,
    }


def train_experiment(
    experiment,
    mode="pilot",
    debug_max_batches=DEBUG_MAX_BATCHES,
):
    print(f"=== {experiment.name} ===")
    print(asdict(experiment))

    is_debug = debug_max_batches is not None

    train_loader, val_loader = make_loaders(experiment)

    model = create_train_model(
        NUM_CLASSES,
        experiment.image_size,
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=experiment.lr,
        weight_decay=experiment.weight_decay,
    )

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=USE_AMP,
    )

    if is_debug:
        # Debug không resume và không ghi đè checkpoint thật.
        resumed_epoch = 0
        best_map50_95 = -1.0
        stale_epochs = 0
        target_epochs = 1

        print(
            f"DEBUG MODE: chạy 1 epoch, "
            f"tối đa {debug_max_batches} batch."
        )
        print("DEBUG MODE sẽ không lưu checkpoint/history.")

    else:
        resumed_epoch, best_map50_95 = (
            load_checkpoint_if_available(
                experiment,
                model,
                optimizer,
                scaler,
            )
        )

        target_epochs = (
            experiment.pilot_epochs
            if mode == "pilot"
            else experiment.max_epochs
        )

        # Khôi phục stale_epochs từ history khi resume.
        previous_history = load_history(experiment)
        stale_epochs = 0

        if (
            not previous_history.empty
            and "stale_epochs" in previous_history.columns
        ):
            previous_stale = previous_history.iloc[-1]["stale_epochs"]

            if pd.notna(previous_stale):
                stale_epochs = int(previous_stale)

    if resumed_epoch >= target_epochs:
        print(
            f"Checkpoint đã ở epoch {resumed_epoch}, "
            f"target của mode='{mode}' là {target_epochs}."
        )
        print("Dùng mode='full' nếu muốn tiếp tục đến max_epochs.")

        del model, optimizer, scaler
        torch.cuda.empty_cache()

        return load_history(experiment)

    for epoch in range(
        resumed_epoch + 1,
        target_epochs + 1,
    ):
        started = time.time()

        train_metrics = train_one_epoch(
            model,
            train_loader,
            optimizer,
            scaler,
            epoch,
            experiment,
            debug_max_batches,
        )

        should_evaluate = (
            is_debug
            or epoch % experiment.eval_every_epochs == 0
            or epoch == target_epochs
        )

        eval_metrics = {
            "mAP50": np.nan,
            "mAP50-95": np.nan,
            "P": np.nan,
            "R": np.nan,
            "F1": np.nan,
            "best_conf_threshold": np.nan,
        }

        if should_evaluate:
            eval_metrics = evaluate_model_checkpointless(
                model,
                val_loader,
                experiment,
                max_batches=3 if is_debug else None,
            )

        current_map = eval_metrics.get(
            "mAP50-95",
            np.nan,
        )

        improved = False

        if should_evaluate and not np.isnan(current_map):
            if (
                current_map
                > best_map50_95
                + experiment.early_stopping_min_delta
            ):
                best_map50_95 = float(current_map)
                stale_epochs = 0
                improved = True
            else:
                stale_epochs += experiment.eval_every_epochs

        row = {
            "experiment": experiment.name,
            "epoch": epoch,
            "target_epochs": target_epochs,
            "elapsed_min": (time.time() - started) / 60.0,
            "evaluated": should_evaluate,
            "best_mAP50-95": best_map50_95,
            "stale_epochs": stale_epochs,
            **train_metrics,
            **eval_metrics,
        }

        print(row)

        if not is_debug:
            append_history(experiment, row)

            # last.pt luôn là trạng thái mới nhất.
            save_checkpoint(
                experiment,
                epoch,
                model,
                optimizer,
                scaler,
                best_map50_95,
                target_name="last.pt",
            )

            # best.pt chỉ cập nhật khi mAP50-95 tốt hơn.
            if improved:
                save_checkpoint(
                    experiment,
                    epoch,
                    model,
                    optimizer,
                    scaler,
                    best_map50_95,
                    target_name="best.pt",
                )

            print_working_disk_usage(
                f"after epoch {epoch}"
            )

        if (
            not is_debug
            and experiment.early_stopping
            and stale_epochs
            >= experiment.early_stopping_patience_epochs
        ):
            print(
                f"Early stopping tại epoch {epoch}; "
                f"best mAP50-95={best_map50_95:.3f}"
            )
            break

    del model, optimizer, scaler, train_loader, val_loader
    torch.cuda.empty_cache()

    if is_debug:
        print("Smoke test hoàn thành, không lưu checkpoint.")
        return pd.DataFrame([row])

    return load_history(experiment)


print("Train functions đã sẵn sàng.")

Train functions đã sẵn sàng.


### 7.1 Baseline - `effdet_d0_baseline_512`

In [ ]:
train_experiment(
    EXPERIMENT_LOOKUP["effdet_d0_baseline_512"],
    mode="pilot",
    debug_max_batches=20,
)

=== effdet_d0_baseline_512 ===
{'name': 'effdet_d0_baseline_512', 'image_size': 512, 'max_epochs': 50, 'pilot_epochs': 20, 'batch_size': 8, 'lr': 0.0002, 'weight_decay': 0.0001, 'eval_every_epochs': 5, 'early_stopping': False, 'early_stopping_patience_epochs': 10, 'early_stopping_min_delta': 0.05, 'use_oversampling': False, 'augmentation_policy': 'baseline'}
DEBUG MODE: chạy 1 epoch, tối đa 20 batch.
DEBUG MODE sẽ không lưu checkpoint/history.


effdet_d0_baseline_512 - epoch 1:   0%|          | 0/810 [00:00<?, ?it/s]

Predicting:   0%|          | 0/203 [00:00<?, ?it/s]

Loading and preparing results...
DONE (t=0.01s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.12s).
Accumulating evaluation results...
DONE (t=0.06s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.001
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.001
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.001
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.001
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.002
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.002
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=1

,experiment,epoch,target_epochs,elapsed_min,evaluated,best_mAP50-95,stale_epochs,train_loss,class_loss,box_loss,train_batches,mAP50,mAP50-95,P,R,F1,best_conf_threshold
0,effdet_d0_baseline_512,1,1,3.002101,True,0.059104,0,1.630533,1.309249,0.006426,20,0.125108,0.059104,0.175055,1.762115,0.318471,0.022778


In [11]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/SH17_outputs/effdet_d0_pytorch"
)

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("Checkpoint và kết quả sẽ lưu tại:")
print(OUTPUT_ROOT)
print("Tồn tại:", OUTPUT_ROOT.exists())

Mounted at /content/drive
Checkpoint và kết quả sẽ lưu tại:
/content/drive/MyDrive/SH17_outputs/effdet_d0_pytorch
Tồn tại: True


In [ ]:
# Baseline pilot trước để kiểm tra toàn bộ pipeline.
RUN_EXPERIMENT_NAMES = ["effdet_d0_baseline_512"]
for name in RUN_EXPERIMENT_NAMES:
    train_experiment(EXPERIMENT_LOOKUP[name], mode="pilot")


=== effdet_d0_baseline_512 ===
{'name': 'effdet_d0_baseline_512', 'image_size': 512, 'max_epochs': 50, 'pilot_epochs': 20, 'batch_size': 8, 'lr': 0.0002, 'weight_decay': 0.0001, 'eval_every_epochs': 5, 'early_stopping': False, 'early_stopping_patience_epochs': 10, 'early_stopping_min_delta': 0.05, 'use_oversampling': False, 'augmentation_policy': 'baseline'}
Không có checkpoint cũ, bắt đầu train từ epoch 1.


effdet_d0_baseline_512 - epoch 1:   0%|          | 0/810 [00:00<?, ?it/s]

{'experiment': 'effdet_d0_baseline_512', 'epoch': 1, 'target_epochs': 20, 'elapsed_min': 34.485673852761586, 'evaluated': False, 'best_mAP50-95': -1.0, 'stale_epochs': 0, 'train_loss': 0.7992224360689705, 'class_loss': 0.5827854958949266, 'box_loss': 0.004328738810080621, 'train_batches': 810, 'mAP50': nan, 'mAP50-95': nan, 'P': nan, 'R': nan, 'F1': nan, 'best_conf_threshold': nan}
saved checkpoint: /content/drive/MyDrive/SH17_outputs/effdet_d0_pytorch/effdet_d0_baseline_512/checkpoints/last.pt
after epoch 1: used=47.22GB free=65.40GB total=112.64GB root=/content


effdet_d0_baseline_512 - epoch 2:   0%|          | 0/810 [00:00<?, ?it/s]

{'experiment': 'effdet_d0_baseline_512', 'epoch': 2, 'target_epochs': 20, 'elapsed_min': 37.79224632581075, 'evaluated': False, 'best_mAP50-95': -1.0, 'stale_epochs': 0, 'train_loss': 0.5608166895162912, 'class_loss': 0.38052263797065355, 'box_loss': 0.003605881021185606, 'train_batches': 810, 'mAP50': nan, 'mAP50-95': nan, 'P': nan, 'R': nan, 'F1': nan, 'best_conf_threshold': nan}
saved checkpoint: /content/drive/MyDrive/SH17_outputs/effdet_d0_pytorch/effdet_d0_baseline_512/checkpoints/last.pt
after epoch 2: used=47.22GB free=65.40GB total=112.64GB root=/content


...

effdet_d0_baseline_512 - epoch 8:   0%|          | 0/810 [00:00<?, ?it/s]

{'experiment': 'effdet_d0_baseline_512', 'epoch': 8, 'target_epochs': 20, 'elapsed_min': 34.05038071870804, 'evaluated': False, 'best_mAP50-95': 18.64639791732667, 'stale_epochs': 0, 'train_loss': 0.3627398728956411, 'class_loss': 0.2391198206095048, 'box_loss': 0.0024724010425550796, 'train_batches': 810, 'mAP50': nan, 'mAP50-95': nan, 'P': nan, 'R': nan, 'F1': nan, 'best_conf_threshold': nan}
saved checkpoint: /content/drive/MyDrive/SH17_outputs/effdet_d0_pytorch/effdet_d0_baseline_512/checkpoints/last.pt
after epoch 8: used=47.32GB free=65.30GB total=112.64GB root=/content


effdet_d0_baseline_512 - epoch 9:   0%|          | 0/810 [00:00<?, ?it/s]

{'experiment': 'effdet_d0_baseline_512', 'epoch': 9, 'target_epochs': 20, 'elapsed_min': 34.025898321469626, 'evaluated': False, 'best_mAP50-95': 18.64639791732667, 'stale_epochs': 0, 'train_loss': 0.3494003064838457, 'class_loss': 0.23055817738727288, 'box_loss': 0.002376842580413745, 'train_batches': 810, 'mAP50': nan, 'mAP50-95': nan, 'P': nan, 'R': nan, 'F1': nan, 'best_conf_threshold': nan}
saved checkpoint: /content/drive/MyDrive/SH17_outputs/effdet_d0_pytorch/effdet_d0_baseline_512/checkpoints/last.pt
after epoch 9: used=47.32GB free=65.30GB total=112.64GB root=/content


effdet_d0_baseline_512 - epoch 10:   0%|          | 0/810 [00:00<?, ?it/s]

In [12]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

from pathlib import Path

last_path = Path(
    "/content/drive/MyDrive/SH17_outputs/"
    "effdet_d0_pytorch/"
    "effdet_d0_baseline_512/"
    "checkpoints/last.pt"
)

print("Checkpoint tồn tại:", last_path.exists())

Mounted at /content/drive
Checkpoint tồn tại: True


In [ ]:
from pathlib import Path
import torch

last_path = Path(
    "/content/drive/MyDrive/SH17_outputs/"
    "effdet_d0_pytorch/"
    "effdet_d0_baseline_512/"
    "checkpoints/last.pt"
)

assert last_path.exists(), f"Không tìm thấy checkpoint: {last_path}"

checkpoint = torch.load(
    last_path,
    map_location="cpu",
    weights_only=False,
)

print("Epoch gần nhất đã hoàn thành:", checkpoint["epoch"])
print("Best mAP50-95 hiện tại:", checkpoint["best_map50_95"])
print("Experiment:", checkpoint["experiment"])

Epoch gần nhất đã hoàn thành: 9
Best mAP50-95 hiện tại: 18.64639791732667
Experiment: effdet_d0_baseline_512


In [ ]:
# Baseline pilot trước để kiểm tra toàn bộ pipeline.
RUN_EXPERIMENT_NAMES = ["effdet_d0_baseline_512"]
for name in RUN_EXPERIMENT_NAMES:
    train_experiment(EXPERIMENT_LOOKUP[name], mode="pilot")


=== effdet_d0_baseline_512 ===
{'name': 'effdet_d0_baseline_512', 'image_size': 512, 'max_epochs': 50, 'pilot_epochs': 20, 'batch_size': 8, 'lr': 0.0002, 'weight_decay': 0.0001, 'eval_every_epochs': 5, 'early_stopping': False, 'early_stopping_patience_epochs': 10, 'early_stopping_min_delta': 0.05, 'use_oversampling': False, 'augmentation_policy': 'baseline'}
resumed checkpoint: /content/drive/MyDrive/SH17_outputs/effdet_d0_pytorch/effdet_d0_baseline_512/checkpoints/last.pt


effdet_d0_baseline_512 - epoch 10:   0%|          | 0/810 [00:00<?, ?it/s]

Predicting:   0%|          | 0/203 [00:00<?, ?it/s]

Loading and preparing results...
DONE (t=0.95s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=14.92s).
Accumulating evaluation results...
DONE (t=3.21s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.212
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.369
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.216
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.001
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.236
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.211
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.345
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.353
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=1

effdet_d0_baseline_512 - epoch 11:   0%|          | 0/810 [00:00<?, ?it/s]

...

{'experiment': 'effdet_d0_baseline_512', 'epoch': 14, 'target_epochs': 20, 'elapsed_min': 46.97722555398941, 'evaluated': False, 'best_mAP50-95': 21.18469531371149, 'stale_epochs': 0, 'train_loss': 0.3009910772979995, 'class_loss': 0.19865259419620773, 'box_loss': 0.0020467696753100574, 'train_batches': 810, 'mAP50': nan, 'mAP50-95': nan, 'P': nan, 'R': nan, 'F1': nan, 'best_conf_threshold': nan}
saved checkpoint: /content/drive/MyDrive/SH17_outputs/effdet_d0_pytorch/effdet_d0_baseline_512/checkpoints/last.pt
after epoch 14: used=60.89GB free=51.73GB total=112.64GB root=/content


effdet_d0_baseline_512 - epoch 15:   0%|          | 0/810 [00:00<?, ?it/s]

Predicting:   0%|          | 0/203 [00:00<?, ?it/s]

Loading and preparing results...
DONE (t=11.34s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=15.05s).
Accumulating evaluation results...
DONE (t=4.81s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.228
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.398
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.234
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.001
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.253
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.219
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.350
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.357
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=

effdet_d0_baseline_512 - epoch 16:   0%|          | 0/810 [00:00<?, ?it/s]

In [13]:
from pathlib import Path
import torch

last_path = Path(
    "/content/drive/MyDrive/SH17_outputs/"
    "effdet_d0_pytorch/"
    "effdet_d0_baseline_512/"
    "checkpoints/last.pt"
)

assert last_path.exists(), f"Không tìm thấy checkpoint: {last_path}"

checkpoint = torch.load(
    last_path,
    map_location="cpu",
    weights_only=False,
)

print("Epoch gần nhất đã hoàn thành:", checkpoint["epoch"])
print("Best mAP50-95 hiện tại:", checkpoint["best_map50_95"])
print("Experiment:", checkpoint["experiment"])

Epoch gần nhất đã hoàn thành: 15
Best mAP50-95 hiện tại: 22.81335913613083
Experiment: effdet_d0_baseline_512


In [14]:
# Baseline pilot trước để kiểm tra toàn bộ pipeline. train từ epoch 16 trở đi vì đã lưu checkpoint từ trước
RUN_EXPERIMENT_NAMES = ["effdet_d0_baseline_512"]
for name in RUN_EXPERIMENT_NAMES:
    train_experiment(EXPERIMENT_LOOKUP[name], mode="pilot")


=== effdet_d0_baseline_512 ===
{'name': 'effdet_d0_baseline_512', 'image_size': 512, 'max_epochs': 50, 'pilot_epochs': 20, 'batch_size': 8, 'lr': 0.0002, 'weight_decay': 0.0001, 'eval_every_epochs': 5, 'early_stopping': False, 'early_stopping_patience_epochs': 10, 'early_stopping_min_delta': 0.05, 'use_oversampling': False, 'augmentation_policy': 'baseline'}
resumed checkpoint: /content/drive/MyDrive/SH17_outputs/effdet_d0_pytorch/effdet_d0_baseline_512/checkpoints/last.pt


effdet_d0_baseline_512 - epoch 16:   0%|          | 0/810 [00:00<?, ?it/s]

{'experiment': 'effdet_d0_baseline_512', 'epoch': 16, 'target_epochs': 20, 'elapsed_min': 40.856856660048166, 'evaluated': False, 'best_mAP50-95': 22.81335913613083, 'stale_epochs': 0, 'train_loss': 0.27035454005738835, 'class_loss': 0.17936199503364386, 'box_loss': 0.0018198508856944555, 'train_batches': 810, 'mAP50': nan, 'mAP50-95': nan, 'P': nan, 'R': nan, 'F1': nan, 'best_conf_threshold': nan}
saved checkpoint: /content/drive/MyDrive/SH17_outputs/effdet_d0_pytorch/effdet_d0_baseline_512/checkpoints/last.pt
after epoch 16: used=60.61GB free=52.01GB total=112.64GB root=/content


effdet_d0_baseline_512 - epoch 17:   0%|          | 0/810 [00:00<?, ?it/s]

{'experiment': 'effdet_d0_baseline_512', 'epoch': 17, 'target_epochs': 20, 'elapsed_min': 37.52650289932887, 'evaluated': False, 'best_mAP50-95': 22.81335913613083, 'stale_epochs': 0, 'train_loss': 0.28458605958723726, 'class_loss': 0.18808229687037292, 'box_loss': 0.0019300752493299912, 'train_batches': 810, 'mAP50': nan, 'mAP50-95': nan, 'P': nan, 'R': nan, 'F1': nan, 'best_conf_threshold': nan}
saved checkpoint: /content/drive/MyDrive/SH17_outputs/effdet_d0_pytorch/effdet_d0_baseline_512/checkpoints/last.pt
after epoch 17: used=60.66GB free=51.96GB total=112.64GB root=/content


...

effdet_d0_baseline_512 - epoch 19:   0%|          | 0/810 [00:00<?, ?it/s]

{'experiment': 'effdet_d0_baseline_512', 'epoch': 19, 'target_epochs': 20, 'elapsed_min': 34.61123791138331, 'evaluated': False, 'best_mAP50-95': 22.81335913613083, 'stale_epochs': 0, 'train_loss': 0.2709313550297125, 'class_loss': 0.1790743929736408, 'box_loss': 0.0018371392395002423, 'train_batches': 810, 'mAP50': nan, 'mAP50-95': nan, 'P': nan, 'R': nan, 'F1': nan, 'best_conf_threshold': nan}
saved checkpoint: /content/drive/MyDrive/SH17_outputs/effdet_d0_pytorch/effdet_d0_baseline_512/checkpoints/last.pt
after epoch 19: used=60.75GB free=51.87GB total=112.64GB root=/content


effdet_d0_baseline_512 - epoch 20:   0%|          | 0/810 [00:00<?, ?it/s]

Predicting:   0%|          | 0/203 [00:00<?, ?it/s]

Loading and preparing results...
DONE (t=10.77s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=12.44s).
Accumulating evaluation results...
DONE (t=2.53s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.236
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.404
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.245
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.001
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.262
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.226
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.359
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.366
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=

### 7.2 Variant 1 - `effdet_d0_tuned_512_es`

In [ ]:
# Variant 1: lr nhẹ hơn, color jitter, early stopping.
RUN_EXPERIMENT_NAMES = ["effdet_d0_tuned_512_es"]
for name in RUN_EXPERIMENT_NAMES:
    train_experiment(EXPERIMENT_LOOKUP[name], mode="pilot")


### 7.3 Variant 2 - `effdet_d0_oversample_minority_512_es`

In [ ]:
# Variant 2: oversample class PPE hiếm.
RUN_EXPERIMENT_NAMES = ["effdet_d0_oversample_minority_512_es"]
for name in RUN_EXPERIMENT_NAMES:
    train_experiment(EXPERIMENT_LOOKUP[name], mode="pilot")


### 7.4 Variant 3 - `effdet_d0_zoom_crop_512_es`

In [ ]:
# Variant 3: random resize/crop cho object nhỏ.
RUN_EXPERIMENT_NAMES = ["effdet_d0_zoom_crop_512_es"]
for name in RUN_EXPERIMENT_NAMES:
    train_experiment(EXPERIMENT_LOOKUP[name], mode="pilot")


## 8. Export Results

In [ ]:
# Ghi manifest cuối để biết notebook đã chạy theo config nào.

run_manifest = {
    "class_names": CLASS_NAMES,
    "minority_class_ids_zero_based": sorted(MINORITY_CLASS_IDS_ZERO_BASED),
    "output_root": str(OUTPUT_ROOT),
    "experiments": [asdict(experiment) for experiment in EXPERIMENTS],
    "run_experiment_names": RUN_EXPERIMENT_NAMES,
    "device": str(DEVICE),
    "use_amp": USE_AMP,
}

(OUTPUT_ROOT / "run_manifest.json").write_text(json.dumps(run_manifest, indent=2))
print(json.dumps(run_manifest, indent=2))
print_working_disk_usage("final")
